# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL as follows:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and columns defined in the Croissant schema.

In [ ]:
# Print all record sets and their fields/columns by @id
# Croissant datasets expose record sets via metadata.recordSet
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_sets = metadata.recordSet
else:
    # If recordSet is empty, mlcroissant will infer from distributions
    record_sets = dataset.record_sets()

record_set_ids = []

print("Record Sets Overview:")
for rs in record_sets:
    recset_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
    record_set_ids.append(recset_id)
    print(f"- Record set @id: {recset_id}")
    fields = rs.get('field', []) if isinstance(rs, dict) else []
    if fields:
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"    - Field @id: {field_id}")
            # Show column if defined
            if isinstance(field, dict) and 'column' in field:
                col = field['column']
                column_id = col['@id'] if isinstance(col, dict) and '@id' in col else col
                print(f"        - Column @id: {column_id}")
    else:
        print("    (No explicit fields defined in schema)")
print()
print("If a record set is not listed, you may inspect its inferred structure with mlcroissant:")
for rset_id in record_set_ids:
    try:
        for rec in dataset.records(record_set=rset_id):
            print(f"Sample record from {rset_id}:")
            pprint.pprint(rec)
            break
    except Exception as e:
        print(f"Could not load records for {rset_id}: {e}")

## 3. Data Extraction
Load data from record set(s) into pandas DataFrames for analysis.
All extraction is referenced by `@id` as per the FAIR^2 schema.

In [ ]:
# List all available record set @id values
print("Available record set @ids:")
print(record_set_ids)

# If only inferred record sets are available, we can try using the dataset's record_sets()
if not record_set_ids:
    # Sometimes mlcroissant will infer record set IDs from distributions
    record_set_ids = dataset.record_sets()
    print("Inferred record set @ids:")
    print(record_set_ids)

# For demo purposes, we'll extract from the first available record set
extracted_dataframes = {}

for record_set_id in record_set_ids:
    # Load records from the record set using its @id
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        extracted_dataframes[record_set_id] = df
        print(f"Data columns for record set {record_set_id}:")
        print(df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Could not extract records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing, and grouping by attributes.
> All fields and columns referenced by their `@id`.

We'll select a numeric field, filter for high values, normalize it, and group by another field.

In [ ]:
# --- Example EDA ---
# We'll select the first DataFrame loaded
if extracted_dataframes:
    first_record_set_id = list(extracted_dataframes.keys())[0]
    df = extracted_dataframes[first_record_set_id]
    print(f"Analyzing DataFrame from record set: {first_record_set_id}")

    # Identify potential numeric fields (by column name containing 'age', 'interval', or similar)
    numeric_col_candidates = [col for col in df.columns if 'Age' in col or 'interval' in col.lower() or 'years' in col.lower() or 'Number' in col]
    if numeric_col_candidates:
        numeric_field_id = numeric_col_candidates[0]
        print(f"Selected numeric field: {numeric_field_id}")

        # Filter for values above a threshold
        threshold = 60
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Select a group field (e.g., Anatomical location or Sex)
        group_field_candidates = [col for col in df.columns if 'location' in col.lower() or 'Sex' in col or 'group' in col]
        if group_field_candidates:
            group_field_id = group_field_candidates[0]
            print(f"Grouping by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print("Grouped data:")
            print(grouped_df.head())
    else:
        print("No numeric field found for demo analysis.")
else:
    print("No DataFrame extracted to perform EDA.")

## 5. Visualization
Visualize data distributions or field relationships using matplotlib or seaborn.

We'll plot the distribution of the selected numeric field and a bar plot for grouped means (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization of numeric field distribution
if extracted_dataframes:
    df = extracted_dataframes[first_record_set_id]
    if numeric_col_candidates:
        numeric_field_id = numeric_col_candidates[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field_id} ({first_record_set_id})")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()

        # Plot grouped mean if group_field_id and grouped_df are defined
        if 'grouped_df' in locals() and group_field_id is not None:
            plt.figure(figsize=(8,4))
            sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
            plt.title(f"Mean {numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.show()


## 6. Conclusion
This notebook loaded and explored the FAIR^2 colorectal cancer dataset package using the `mlcroissant` library. 
Key steps included referencing entities via their `@id`, extracting records as DataFrames, and visualizing numeric field distributions and group means. 

The dataset enables deeper clinical analysis of second primary colorectal cancer in survivors, supporting biomarker stratification and anatomical distribution studies. Future applications include developing predictive models and further domain-specific analyses using the FAIR^2 schema.